# Task B -- TAPT with Task A training text

This notebook tests whether additional mixed Kannada-English comments improve Task B
through masked-language-model adaptation. It adds the **text** from
`binary_train.csv` to the normal Task B TAPT corpus. Task A labels are never read or
used.

The Task A training file is explicitly permitted for this experiment. It contains
some Task B validation comments, so this is a deliberate transductive run; the
`--allow-transductive` flag records that choice in the log. Do not interpret its
CodaBench score as a leak-free benchmark.

| setting | value |
|---|---|
| TAPT corpus | Task B train + OffensEval Kannada + Task A training comments |
| TAPT labels | none; comments only |
| classifier | all 3,159 Task B rows, no deduplication |
| reinitialization | one final MuRIL encoder layer |
| R-Drop | 0.5 |
| seeds | 42, 43, 44, 45, 46; validation probabilities averaged |

Expected runtime is approximately 3--4 hours on a T4 x2 or P100. Use GPU and
Internet, then **Save Version -> Save & Run All**. The output is a CodaBench ZIP;
there is no honest local F1 for this full-data fit.

In [ ]:
import os, pathlib, re, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Run expanded TAPT

The Task A CSV is passed as a text source only. `corpora.load` detects the comment
column and ignores the `Hate`/`Non-Hate` labels. The explicit transductive flag is
required because the file contains released Task B validation comments.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-taska-text"
TAPT_LOG = "artifacts/logs/tapt_taska_text.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing expanded TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", "data/raw/multiclass_train.csv",
                    "data/external/offenseval_kn.csv",
                    "data/raw/binary_train.csv",
         "--allow-transductive",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe",
         "--epochs", "8", "--out", TAPT_OUT], log=TAPT_LOG)
assert (pathlib.Path(TAPT_OUT) / "config.json").exists(), "TAPT checkpoint was not written"
if pathlib.Path(TAPT_LOG).exists():
    tapt_text = pathlib.Path(TAPT_LOG).read_text()
    assert "TRANSDUCTIVE: admitted" in tapt_text, "Task A text was not admitted"
    assert "binary_train.csv" in tapt_text, "Task A corpus is missing from the log"
print("expanded TAPT checkpoint ready:", TAPT_OUT)

## 2. Train the full-data Task B classifier

`--folds 1` trains each seed on every labelled Task B row. This is the same
one-layer MuRIL + R-Drop recipe used by the confirmed Task B submission; the only
intended change is the expanded TAPT corpus.

In [ ]:
TAG = "b_tapt_taska_full"
TRAIN_LOG = pathlib.Path("artifacts/logs") / f"{TAG}.log"
RUN_DIR = pathlib.Path("artifacts/runs") / TAG
run([sys.executable, "-u", "-m", "hastika.task_b.train",
     "--tag", TAG, "--model", TAPT_OUT,
     "--folds", "1", "--no-dedupe",
     "--reinit-layers", "1", "--rdrop", "0.5",
     "--aux-weight", "0",
     "--seeds", "42", "43", "44", "45", "46",
     "--epochs", "6"], log=str(TRAIN_LOG))
log_text = TRAIN_LOG.read_text()
fits = re.findall(r"===== seed (\d+) FULL FIT, (\d+) rows, no validation =====", log_text)
assert [seed for seed, _ in fits] == ["42", "43", "44", "45", "46"], fits
assert all(rows == "3159" for _, rows in fits), fits
assert "reinit=1" in log_text, "one-layer reinitialization was not enabled"
assert (RUN_DIR / "test_probs.npy").exists(), "validation probabilities were not written"
assert (RUN_DIR / "predictions.csv").exists(), "predictions were not written"
print("five expanded-TAPT full-data fits completed:", RUN_DIR)

## 3. Package the CodaBench submission

The helper validates the 395 validation IDs, the `id,label` header and the six
allowed Task B labels before writing a ZIP containing one bare `predictions.csv`.

In [ ]:
import pandas as pd

PRED = RUN_DIR / "predictions.csv"
ZIP = pathlib.Path("/kaggle/working/b_tapt_taska_full.zip")
run([sys.executable, "-m", "hastika.common.submission",
     "--task", "b", "--pred", str(PRED), "--out", str(ZIP)])
assert ZIP.exists(), "submission ZIP was not written"
with zipfile.ZipFile(ZIP) as z:
    assert z.namelist() == ["predictions.csv"], z.namelist()
pred = pd.read_csv(PRED)
assert list(pred.columns) == ["id", "label"]
assert len(pred) == 395 and pred["id"].is_unique
print("READY TO UPLOAD:", ZIP)

## 4. Preserve reproducibility files

Download the ZIP and this output directory from Kaggle. Submit only
`b_tapt_taska_full.zip` to the Task B validation phase.

In [ ]:
OUT = pathlib.Path("/kaggle/working/tapt_taska_full_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for source in [ZIP, PRED, RUN_DIR / "test_probs.npy", TRAIN_LOG, pathlib.Path(TAPT_LOG)]:
    if source.exists():
        shutil.copy2(source, OUT / source.name)
print("download:", OUT)
print("files:", sorted(x.name for x in OUT.iterdir()))